# Floor Plan Recognition - Baseline Training (YOLOv8n)

**目的**: Roboflow データセットを用いて YOLOv8n で物体検出のベースラインを学習する

**実行環境**: Google Colab (T4 GPU)

**所要時間**: 全工程で約 30-60 分(うち学習が 15-30 分)

---

## 事前準備(初回のみ)

Colab Secrets で `ROBOFLOW_API_KEY` を登録しておく必要があります。

1. 左サイドバーの 🔑 (鍵アイコン) をクリック
2. 「新しいシークレットを追加」
3. 名前: `ROBOFLOW_API_KEY`、値: あなたの API キー
4. 「ノートブックのアクセス権」をオンにする

## Section 1: 環境セットアップ

### 1.1 GPU の確認

In [ ]:
!nvidia-smi

GPU が `Tesla T4` などと表示されれば OK。
もし "command not found" などのエラーが出る場合は、Colab メニューの「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選択してください。

### 1.2 リポジトリの clone

In [ ]:
import os

# Colab の作業ディレクトリ
WORKDIR = "/content/floor-plan-recognition"

if not os.path.exists(WORKDIR):
    !git clone https://github.com/Mao925/floor-plan-recognition.git {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

### 1.3 必要なライブラリのインストール

In [ ]:
# 学習に必要な最小限のライブラリ
# (Colab には torch / opencv / pillow / pyyaml は標準で入っている)
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
# バージョン確認
import torch
import ultralytics

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: データ準備

### 2.1 Roboflow API キーを Colab Secrets から取得

In [ ]:
from google.colab import userdata

# Colab Secrets から API キーを取得
try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print(f"✅ API キー取得成功: {ROBOFLOW_API_KEY[:3]}***{ROBOFLOW_API_KEY[-3:]}")
except Exception as e:
    print(f"❌ エラー: {e}")
    print("\nColab Secrets に ROBOFLOW_API_KEY を登録してください")
    print("画面左の 🔑 アイコン → 新しいシークレットを追加")

### 2.2 Roboflow からデータセットをダウンロード

In [ ]:
# .env を Colab 側に書き出して、スクリプトがそのまま使えるようにする
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

# データセット取得スクリプトを実行
!python scripts/download_roboflow.py

### 2.3 6クラスへの前処理(クラスフィルタリング + 再分割)

In [ ]:
!python scripts/prepare_dataset.py

In [ ]:
# data.yaml の中身を確認
!cat data/floorplan_yolo/data.yaml

## Section 3: モデル学習(YOLOv8n ベースライン)

### 3.1 学習設定

**ベースライン設定の意図**:
- モデル: `yolov8n.pt` (もっとも小さい・速い) → 最初のベースラインに最適
- エポック数: 50 (データが小規模なので過学習リスクあり、EarlyStopping で調整)
- 画像サイズ: 640 (YOLO 標準)
- バッチサイズ: 16 (T4 GPU で安全な値)

ここで重要なのは「ベースラインとして適切な値を選ぶこと」であって「最初から完璧を目指す」ことではない。

In [ ]:
from ultralytics import YOLO

# 事前学習済み YOLOv8n をロード
model = YOLO('yolov8n.pt')

# 学習設定
results = model.train(
    data='data/floorplan_yolo/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='baseline_yolov8n',
    project='runs/detect',
    patience=15,                # 15 epoch 改善がなければ早期終了
    save=True,
    plots=True,                 # 学習曲線などのグラフを自動生成
    device=0,                   # GPU 使用
    seed=42,                    # 再現性確保
)

print("\n✅ 学習完了")

### 3.2 学習結果のフォルダ構成を確認

In [ ]:
!ls -la runs/detect/baseline_yolov8n/
print("\n--- weights ---")
!ls -la runs/detect/baseline_yolov8n/weights/

## Section 4: 評価

### 4.1 学習曲線とメトリクスの確認

In [ ]:
# Ultralytics が自動生成した結果画像を表示
from IPython.display import Image, display

results_dir = 'runs/detect/baseline_yolov8n'

for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = f'{results_dir}/{img_name}'
    if os.path.exists(img_path):
        print(f"\n=== {img_name} ===")
        display(Image(img_path))

### 4.2 test セットでの最終評価

学習中の val 評価とは別に、未使用の test セットで最終評価する。これが「実運用時に近い性能」の指標。

In [ ]:
# 学習済みベストモデルを読み込み
best_model = YOLO(f'{results_dir}/weights/best.pt')

# test セットで評価
test_metrics = best_model.val(
    data='data/floorplan_yolo/data.yaml',
    split='test',
    name='baseline_test_eval',
    project='runs/detect',
)

print("\n=== Test セット全体メトリクス ===")
print(f"mAP@0.5:        {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {test_metrics.box.map:.4f}")
print(f"Precision:      {test_metrics.box.mp:.4f}")
print(f"Recall:         {test_metrics.box.mr:.4f}")

print("\n=== クラス別 mAP@0.5 ===")
class_names = ['door', 'shower', 'sink', 'staircase', 'toilet', 'window']
for i, name in enumerate(class_names):
    ap = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0
    print(f"  {name:12s}: {ap:.4f}")

### 4.3 推論サンプルの可視化

In [ ]:
import random
from pathlib import Path
from IPython.display import Image, display

# test セットからランダムに4枚
test_images = sorted(Path('data/floorplan_yolo/test/images').glob('*.jpg'))
random.seed(42)
samples = random.sample(test_images, min(4, len(test_images)))

# 推論
predict_results = best_model.predict(
    source=[str(p) for p in samples],
    save=True,
    project='runs/detect',
    name='baseline_samples',
    conf=0.25,
)

# 結果を表示
pred_dir = Path('runs/detect/baseline_samples')
for img_path in pred_dir.glob('*.jpg'):
    print(f"\n=== {img_path.name} ===")
    display(Image(str(img_path)))

## Section 5: 結果の保存

### 5.1 学習済みモデルとログを zip にまとめる

ローカル(M2 Mac)に持ち帰って推論用に使うため、必要なファイルをまとめる。

In [ ]:
import shutil

# 持ち帰るもの: best.pt と学習結果の画像・メトリクス
src = 'runs/detect/baseline_yolov8n'
out_zip = '/content/baseline_yolov8n_results.zip'

shutil.make_archive(out_zip.replace('.zip', ''), 'zip', src)
print(f"✅ Zip 作成完了: {out_zip}")
!ls -lh {out_zip}

### 5.2 ダウンロード

次のセルを実行すると Colab からブラウザにダウンロードが始まる。ダウンロードしたら、ローカルの `floor-plan-recognition/models/` に置く。

In [ ]:
from google.colab import files
files.download(out_zip)

---

## 次のステップ

ベースラインができたら、以下の仮説検証サイクルに進む:

1. **モデルサイズの比較**: YOLOv8n vs YOLOv8s vs YOLOv8m → 精度と速度のトレードオフ
2. **入力解像度の影響**: 640 vs 1024 → 小物体検出の改善
3. **データ拡張の効果**: 標準 vs より強い拡張 → 汎化性能
4. **SAHI(Slicing Aided Hyper Inference)**: 推論時のタイル分割で小物体改善

各実験の前後で「仮説 → 実験 → 結果 → 解釈」を README に追記していく。